# Tarea 6: Hierarchical Attention

Procesamiento de Lenguaje Natural

Eric Lemus Avalos 

In [1]:
import os 
import re
import time 
import random
import copy 
from argparse import Namespace

import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter 
import xml.etree.ElementTree as ET


import torch 
import torch.nn.functional as F
from torch import nn, optim
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence, pad_packed_sequence, pack_padded_sequence
from tqdm.auto import tqdm

from sklearn.metrics import f1_score

import nltk 
# nltk.download('punkt')

In [2]:
os.getcwd()

'C:\\Users\\ericl\\tareas_nlp\\tarea_6'

In [3]:
args = Namespace()
args.train_path = r'./es_train'
args.train_truth = r'./es_train/truth.txt'

args.val_path = r'./es_val'
args.val_truth = r'./es_val/truth.txt'

## 1. Preprocesamiento de los datos

Definimos las siguientes funciones para extraer y hacer una limpieza del texto. 

In [4]:
def load_truth_values(truth_file):
    """
    Función que lee el archivo truth.txt y extrae los valores de cada usuario en un diccionario.
    """
    truth_dict = {} 
    
    with open(truth_file, "r", encoding="utf-8") as f:
        for linea in f:
            partes = linea.strip().split(":::") 
            id_usuario, genero, nacionalidad = partes
            truth_dict[id_usuario] = {"genero": genero, "nacionalidad": nacionalidad}
            
    return truth_dict


def extract_tweets_from_xml(xml_file):
    """
    Función que extrae los tweets de un archivo XML.
    """
    tweets = []
    try:
        tree = ET.parse(xml_file)
        root = tree.getroot()
        documents = root.find("documents")
        for doc in documents.findall("document"):
            tweet = doc.text
            tweets.append(tweet.strip())
    
    except:
        print(f"error con: {xml_file}")
    
    return tweets

def load_all_tweets(path_train, truth_dict):
    """
    Función que recorre todos los archivos XML en path_train, extrae los tweets y 
    construye un DataFrame con las columnas 'id_usuario', 'tweet', 'genero' y 'nacionalidad'.
    """
    registros = []
    
    for archivo in os.listdir(path_train):
        if archivo.endswith(".xml"):
            id_usuario = os.path.splitext(archivo)[0]
            xml_file = os.path.join(path_train, archivo)
            tweets = extract_tweets_from_xml(xml_file)
            
            genero = truth_dict.get(id_usuario, {}).get("genero")
            nacionalidad = truth_dict.get(id_usuario, {}).get("nacionalidad")
            
            for tweet in tweets:
                registros.append({
                    "id_usuario": id_usuario,
                    "tweet": tweet,
                    "genero": genero,
                    "nacionalidad": nacionalidad
                })
                
    df = pd.DataFrame(registros, columns=["id_usuario", "tweet", "genero", "nacionalidad"])
    return df

In [5]:
truth_dict_train = load_truth_values(args.train_truth)
truth_dict_val = load_truth_values(args.val_truth)

In [7]:
%%time
df_train = load_all_tweets(args.train_path, truth_dict_train)

CPU times: total: 625 ms
Wall time: 851 ms


In [8]:
%%time
df_val = load_all_tweets(args.val_path,truth_dict_val)

CPU times: total: 0 ns
Wall time: 565 ms


In [9]:
def corpus_analisis(df, col='tweet'):

    corpus = df[col].tolist()
    all_text = " ".join(corpus)
    tokens = all_text.split()
    
    #print("Primeros 20 tokens:", tokens[:20])
    total_palabras = len(tokens)
    print(f"\nPalabras totales: {total_palabras}")
    vocabulario = len(set(tokens))
    print(f"Vocabulario (tokens únicos): {vocabulario}")
    
    def lexical_diversity(text):
        return len(text) / len(set(text))
    
    riqueza = lexical_diversity(tokens)
    print(f"Riqueza del vocabulario: {riqueza:.2f}")

corpus_analisis(df_train, col='tweet')


Palabras totales: 4436204
Vocabulario (tokens únicos): 608281
Riqueza del vocabulario: 7.29


### __Limpieza de los tweets__

In [10]:
def split_hashtag(tag):
    """
    Separa un hashtag en palabras.
    Utiliza una expresión regular para extraer secuencias en formato camel case o dígitos.
    Si el hashtag comienza con dígitos seguidos de una sola letra, se unen.
    """
    tag = tag.lstrip('#')
    parts = re.findall(r'[A-Z]?[a-z]+|[A-Z]+(?![a-z])|\d+', tag)
    if len(parts) >= 2 and parts[0].isdigit() and len(parts[1]) == 1:
        parts[0] = parts[0] + parts[1]
        parts.pop(1)
    return ' '.join(parts).lower()


def clean_tweet(tweet):
    tweet = re.sub(r'http\S+|www\S+|https\S+', '', tweet)
    tweet = re.sub(r'\.\.\.', '', tweet)
    tweet = re.sub(r'\.\.', ' ', tweet)
    tweet = re.sub(r'\s+', ' ', tweet).strip()
    tweet = re.sub(r"[!?\¿\\`¡]()", "", tweet)
    tweet = tweet.replace('-', ' ')
    
    def normalize_laughter(word):
        lw = word.lower()
        if set(lw) <= {'j', 'a'} and len(lw) >= 2:
            return "jaja"
        elif set(lw) <= {'j', 'e'} and len(lw) >= 2:
            return "jeje"
        elif set(lw) <= {'h', 'a'} and len(lw) >= 2:
            return "haha"
        else:
            return word
    
    tokens = tweet.split()
    tokens = [normalize_laughter(word) for word in tokens]
    tweet = " ".join(tokens)


    processed_tokens = []
    for word in tweet.split():
        if word.startswith('#'):
            processed_tokens.append(split_hashtag(word))
        elif word.startswith('@'):
            processed_tokens.append(word.lstrip('@').lower())
        else:
            processed_tokens.append(word)
    tweet = " ".join(processed_tokens)
    
    return tweet.lower()



In [11]:
muestra = df_val.iloc[2000]['tweet']
print(f'{muestra}\n')
tweet_limpio = clean_tweet(muestra)
print(tweet_limpio)

Ay la mujer dándole el ramo de flores AJAJJAJAJAJAJJA ME MEO 😂😂 #PantojaEH

ay la mujer dándole el ramo de flores jaja me meo 😂😂 pantoja eh


In [157]:
# del df_train["preprocess_tweet"]
# del df_val["preprocess_tweet"]

In [12]:
df_train["preprocess_tweet"] = df_train['tweet'].apply(clean_tweet)
df_val["preprocess_tweet"] = df_val['tweet'].apply(clean_tweet)

In [13]:
corpus_analisis(df_train, col='preprocess_tweet')


Palabras totales: 4378909
Vocabulario (tokens únicos): 346024
Riqueza del vocabulario: 12.65


In [48]:
#save data
output_dir = os.path.join("preprocess_data")
os.makedirs(output_dir, exist_ok=True)
df_train.to_csv(os.path.join(output_dir, "df_train.csv"), index=False)
df_val.to_csv(os.path.join(output_dir, "df_val.csv"), index=False)

In [16]:
args.train = r'./preprocess_data/df_train.csv'
args.val = r'./preprocess_data/df_val.csv'
args.split = 'train'

### __Dataset pytorch__

In [14]:
class DatasetAutorProf(Dataset): 
    def __init__(self, args):
        super(Dataset,self).__init__()
        if args.split == 'train':
            self.load_dataset(args.train)
        elif args.split == 'val':
            self.load_dataset(args.val)
        else:
            raise ValueError("Invalid split argument.")
        self.Create_vocab_load_embeddings()

    def __len__(self):
        return len(self.df_grouped)

    def __getitem__(self,index):
        label = self.df_grouped.iloc[index]['nacionalidad']
        senteces, sents_ids = self.preprocess_text(index)
        return senteces, sents_ids, label

    
    def load_dataset(self, path_csv):
        """Carga el archivo cv con los tweets limpios."""
        df = pd.read_csv(path_csv)
        
        self.label_map = {'venezuela': 0,  'peru': 1,  'spain': 2, 'mexico': 3, 
                          'colombia': 4,  'argentina': 5,  'chile': 6}
        
        df['nacionalidad'] = df['nacionalidad'].str.lower().map(self.label_map)
        
        self.df_grouped = df.groupby('id_usuario').agg({\
                                                        'preprocess_tweet': list,      
                                                        'nacionalidad': 'first'   
                                                        }).reset_index()
        
        # Creamos la lista [[sentencias del usuario], id_usuario]
        return self.df_grouped

    def preprocess_text(self, index):
        sentences = self.df_grouped.iloc[index]['preprocess_tweet']
        sents_tokenized = [nltk.word_tokenize(str(text)) for text in sentences]
        sents_ids = [[self.vocab[word] if word in self.vocab.keys() else 1 for word in sentence]\
                     for sentence in sents_tokenized]
        return sents_tokenized, sents_ids

        
    def Create_vocab_load_embeddings(self): 
        '''Embeddings preentrenados en twitter.
           emb_mat: Matriz de embeddings. Un vector de tamaño 200 para cada palabra del vocabulario.
           vocab: Diccionario, asigna a cada palabra su renglón correspondiente en la matriz de embeddings.
        '''
        embeddings_list = []
        self.vocab_dict = {}
        self.vocab = {}
        with open('./word2vec_col.txt', 'r', encoding='utf8', errors='replace') as f:
            for i, line in enumerate(f):
                if i == 0:
                    continue
                values = line.strip().split()
                word = values[0]
                vector = np.asarray(values[1:], dtype='float32') 
                self.vocab_dict[i+1] = word
                self.vocab[word] = i+1
                embeddings_list.append(vector)
        embeddings_list.insert(0,np.mean(np.vstack(embeddings_list), axis=0))
        embeddings_list.insert(0,np.zeros(100))
        self.vocab_dict[0] = '[PAD]'
        self.vocab_dict[1] = '[UNK]'
        self.vocab['[PAD]'] = 0
        self.vocab['[UNK]'] = 1
        self.emb_mat = np.vstack(embeddings_list)
        
        return self.vocab, self.emb_mat
        
    def collate_fn(self, batch):
        """
        Función para generar batches de manera dinámica.
        
        Cada muestra es una tupla: (sentences, sents_ids, label)
          - sentences: lista de oraciones tokenizadas (listas de tokens)
          - sents_ids: lista de listas de IDs (cada lista es una oración)
          - label: entero
        """
        batch_sentences, batch_sents_ids, batch_labels = zip(*batch)
        
        max_num_sents = 100
        max_token_len = 0 # Longitud maxima de tokens entre las oraciones del batch
        for sents_ids in batch_sents_ids:
            for ids in sents_ids:
                max_token_len = max(max_token_len, len(ids))
        
        # Procesar cada muestra del batch
        processed_samples = []         
        processed_original_sentences = []  
        for sents_ids, sentences in zip(batch_sents_ids, batch_sentences):
            sample_tensor = []  # Almacenará los tensores de cada oración en la muestra
            sample_tokens = []  # Almacenará los tokens de la muestra 
            
            # Procesar cada oración de la muestra
            for ids, tokens in zip(sents_ids, sentences):
                ids_tensor = torch.tensor(ids, dtype=torch.long)
                
                # Si la oración tiene menos tokens que el máximo, se hace padding
                if len(ids) < max_token_len:
                    pad = torch.zeros(max_token_len - len(ids), dtype=torch.long)
                    ids_tensor = torch.cat([ids_tensor, pad])
                sample_tensor.append(ids_tensor)
                sample_tokens.append(tokens)
            
            # Si la muestra tiene menos oraciones que max_num_sents, se añade padding a nivel de oraciones
            if len(sample_tensor) < max_num_sents:
                for _ in range(max_num_sents - len(sample_tensor)):
                    sample_tensor.append(torch.zeros(max_token_len, dtype=torch.long))
                    sample_tokens.append([])
            
            # Convertir la lista de oraciones en un tensor (dim: num_oraciones x max_token_len)
            sample_tensor = torch.stack(sample_tensor)
            processed_samples.append(sample_tensor)
            processed_original_sentences.append(sample_tokens)
        
        # Stackear todas las muestras para formar el batch
        batch_tensor = torch.stack(processed_samples)  # (batch_size, max_num_sents, max_token_len)
        labels = torch.tensor(batch_labels, dtype=torch.long)
        
        return batch_tensor, labels, processed_original_sentences

In [17]:
%%time
args.split = 'train'
dataset_train = DatasetAutorProf(args)

CPU times: total: 13.2 s
Wall time: 17.3 s


In [18]:
%%time
args.split = 'val'
dataset_val = DatasetAutorProf(args)

CPU times: total: 12.5 s
Wall time: 16.7 s


In [19]:
dataset_train.df_grouped

,id_usuario,preprocess_tweet,nacionalidad
0,1011b55a13502bcb7562d610a05ef3bd,[4f rebelion de patriotas rebelión cívico mili...,0
1,1026000a7c186f3a4aba1053f9956b42,"[mongodoy es un pobre diablo, un dia nos sorpr...",1
2,1032e9d0da6d21ceab370759d38930a9,[que buen orador es eduardo torres dulce. es c...,2
3,10880d19d7346373a42b18e505bbc4d5,[nueva orden ejecutiva de trump también afecta...,0
4,10aef88d048dafbc75639241b9e35df8,[me gustó un video de youtube cómo hacer un pa...,3
...,...,...,...
3355,ff91e621bd80e9c980a6e7f8550a1d80,"[danielpistola bueno, pero agarra un marcador ...",0
3356,ffb5c29c835e3cecbdaa97bfea5bbe3b,[que se haga tu voluntad y no la mía señor se ...,4
3357,ffc9c0b137c5bbb3f9173e7af991e122,[cómo alguien no se va a enamorar de este ange...,6
3358,ffebe1735cd1f0e69d8210376a9dc377,"[us$20 millones sagrados, educaci n con respet...",1


### __DataLoader__

In [27]:
args.input_size = 100
args.hidden_size = 128
args.num_layers =  1
args.dense_hidden_size = 128
args.bidirectional = True
args.batch_size = 4
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [28]:
train_loader = DataLoader(dataset_train, 
                          batch_size=args.batch_size, 
                          collate_fn = dataset_train.collate_fn, 
                          shuffle=True)

val_loader = DataLoader(dataset_val, 
                        batch_size=args.batch_size, 
                        collate_fn = dataset_val.collate_fn, 
                        shuffle=False)

In [29]:
batch = next(iter(train_loader))
print(f'X (batch_tensor) shape: {batch[0].shape}')   # (batch_size, max_num_sents, max_token_len)
print(f'y (labels) shape: {batch[1].shape}')
# print(f'Senteces preprocessed : {batch[2]}') 

X (batch_tensor) shape: torch.Size([4, 100, 31])
y (labels) shape: torch.Size([4])


En este paso hicimops una limpieza de los tweets por usuario (funcion ``clean_tweet``), los cuales procesamos en la clase ``DatasetAutorProf``, para formar los batches primero encontramos la longitud máxima de tokens entre las oraciones del batch y realizamos padding si es necesario y retornamos nuestro batch en forma de tensor asi como las etiquetas y las oraciones originales.

## 2. RNN con atención en jerárquia

Considerando el paper _Hierarchical Attention Networks for Document Classification_ realizaremos una implementacion de una GRU (bidreccional) para codificar las palabras (Word Encode). Creamos un modulo de atencion que sigue la siguiente estrucutura: 


<!-- $$
\begin{align*}
x_{it} &= W_e w_{it}, \quad t \in [1,T], \\
\overrightarrow{h}_{it} &= \overline{\textrm{GRU}}(x_{it}), \quad t \in [1,T], \\
\overleftarrow{h}_{it} &= \overleftarrow{\textrm{GRU}}(x_{it}), \quad t \in [T,1].
\end{align*}
$$ -->

$$
u_i = \tanh(W h_i + b),
$$

$$
\alpha_i = \frac{\exp(u_i^\top u)}{\sum_i \exp(u_i^\top u)},
$$

$$
v = \sum_i \alpha_i h_i.
$$


<!-- $$
\mathbf{c}_i = \sum_{j=1}^{T_x} \alpha_{ij} \mathbf{h}_j.
$$



$$
\alpha_{ij} = \frac{\exp(e_{ij})}{\sum_{k=1}^{T_x} \exp(e_{ik})},
$$

$$
e_{ij} = a(s_{i-1}, h_j)
$$

Donde $e_{ij}$ scores que tanto hacen match los inputs al rededor del estado oculto. 

Para nuestro caso consideramos la función $e_{ij}$ es aprendida por una red neuronal. 

$$
e_{t} = \tanh(Wh_{t}+b) 
$$

para calcular los scores entonces usamos  $(e_{t} \cdot e_{t}) = e_{t}^Te_{t}$, por lo que el vector de contexto lo calculamos como


$$
\mathbf{c} = \sum_t \alpha_t h_t
$$ -->

In [31]:
class AttnModule(nn.Module):
    def __init__(self, args,input_size):
        super(AttnModule, self).__init__()
        self.fc1 = nn.Linear(in_features=input_size, out_features=args.attn_hidden_size)
        self.fc2 = nn.Linear(in_features=args.attn_hidden_size, out_features=1, bias=False)

    def forward(self, hidden_outputs):
        u = torch.tanh(self.fc1(hidden_outputs))
        alpha = F.softmax(self.fc2(u), dim = 1)
        v = torch.sum(alpha * hidden_outputs, dim = 1)
        return v, alpha.squeeze(-1)

In [30]:
class HAN(nn.Module):
    def __init__(self, args, embedding_matrix=None):
        super(HAN, self).__init__()
        
        self.args = args
        
        # Embedding preentrenada
        self.Embedding = nn.Embedding.from_pretrained(
            embeddings=torch.FloatTensor(embedding_matrix), 
            freeze=False)

        # Word-level GRU
        self.word_GRU = nn.GRU(
            input_size=args.input_size, 
            hidden_size=args.hidden_size, 
            num_layers=args.num_layers, 
            bidirectional=args.bidirectional,
            batch_first=True)
        
        # atención a nivel de palabra
        if self.args.use_word_attn:
            self.word_atten = AttnModule(args, input_size=args.hidden_size * 2)
        
        # Sentence-level GRU
        self.sent_GRU = nn.GRU(
            input_size=args.hidden_size * 2, 
            hidden_size=args.hidden_size, 
            num_layers=args.num_layers, 
            bidirectional=args.bidirectional,
            batch_first=True)
        
        # atención a nivel de oración
        if self.args.use_sentence_attn:
            self.sent_atten = AttnModule(args, input_size=args.hidden_size * 2)
        
        # Clasificador
        self.classifier = nn.Sequential(
            nn.Linear(args.hidden_size * 2, args.dense_hidden_size),
            nn.BatchNorm1d(args.dense_hidden_size),
            nn.ReLU(),
            nn.Linear(args.dense_hidden_size, 7, bias=False)
        )

    def forward(self, x):
        # x shape: (batch_size, num_sents, token_length)
        batch_size, num_sents, token_length = x.size()
        
        # Aplanar para word-level RNN
        x = x.view(-1, token_length)  # (batch_size * num_sents, token_length)
        x = self.Embedding(x)         # (batch_size * num_sents, token_length, emb_dim)
        
        # RNN a nivel de palabra
        word_output, _ = self.word_GRU(x)  # (batch_size * num_sents, token_length, hidden_size*2)
        
        # Aplicar atención de palabra si corresponde, sino pooling
        if self.args.use_word_attn:
            v, word_scores = self.word_atten(word_output)  #  v: (batch_size * num_sents, hidden_size*2)
        else:
            # average pooling a lo largo de los tokens
            v = torch.mean(word_output, dim=1) 
            word_scores = None
        
        # “Des-aplanar” para oraciones
        v_word = v.view(batch_size, num_sents, -1)  #  (batch_size, num_sents, hidden_size*2)
        
        # RNN a nivel de oración
        sent_output, _ = self.sent_GRU(v_word)  #  (batch_size, num_sents, hidden_size*2)
        
        # Aplicar atención de oración si corresponde, sino pooling
        if self.args.use_sentence_attn:
            v_sen, sent_scores = self.sent_atten(sent_output)  # v_sen: (batch_size, hidden_size*2)
        else:
            # Pooling, mean pooling de oraciones
            v_sen = torch.mean(sent_output, dim=1) 
            sent_scores = None
        
        # Clasificador 
        final_output = self.classifier(v_sen)  #(batch_size, 7)
        
        return final_output, word_scores, sent_scores


### __Hiperparámetros__ 

In [32]:
args.lr = 0.001
args.num_epochs = 3
args.weight_decay=0.0001
args.beta1=0.1
args.beta2=0.999
args.attn_hidden_size = 50
args.use_word_attn = True     
args.use_sentence_attn = True   

In [33]:
model = HAN(args, embedding_matrix=dataset_train.emb_mat).to(device)
optimizer = optim.Adam(model.parameters(), 
                       lr=args.lr,
                       weight_decay=args.weight_decay, 
                       betas = (args.beta1, args.beta2))
criterion = nn.NLLLoss()

In [34]:
model

HAN(
  (Embedding): Embedding(973267, 100)
  (word_GRU): GRU(100, 128, batch_first=True, bidirectional=True)
  (word_atten): AttnModule(
    (fc1): Linear(in_features=256, out_features=50, bias=True)
    (fc2): Linear(in_features=50, out_features=1, bias=False)
  )
  (sent_GRU): GRU(256, 128, batch_first=True, bidirectional=True)
  (sent_atten): AttnModule(
    (fc1): Linear(in_features=256, out_features=50, bias=True)
    (fc2): Linear(in_features=50, out_features=1, bias=False)
  )
  (classifier): Sequential(
    (0): Linear(in_features=256, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Linear(in_features=128, out_features=7, bias=False)
  )
)

### __Evaluación__

In [35]:
def eval_model(model, dataloader, criterion, device):
    '''Función para evaluar el modelo.'''
    with torch.no_grad():
        model.eval()
        
        losses = []
        preds = torch.empty(0).long()
        targets = torch.empty(0).long()
        word_scores_list = []
        sent_scores_list = []
        sent_list = []
        pred_list = []
        labels_list = []
        
        for data in tqdm(dataloader):
            torch.cuda.empty_cache()
            batch_tensor, labels, sentences = data
            
            batch_tensor, labels = batch_tensor.to(device), labels.to(device)
            # Obtenemos la salida sin procesar (logits)
            logits, word_scores, sent_scores = model(batch_tensor)
            output = F.log_softmax(logits, dim=1)
            
            # Calculamos la pérdida usando los logits directamente
            loss = criterion(output, labels)
            losses.append(loss.item())
            
            # Luego obtenemos las predicciones para las métricas
            predictions = F.log_softmax(output, dim=1).argmax(1)
            preds = torch.cat([preds, predictions.cpu()], dim=0)
            targets = torch.cat([targets, labels.cpu()], dim=0)

            # Si se necesitan las puntuaciones de atención, se pueden procesar
            if word_scores is not None:
                # Convertir los tensores de atención a listas (opcional)
                wordAtt_scores = word_scores.cpu().detach().tolist()
                sentAtt_scores = sent_scores.cpu().detach().tolist()
                word_scores_list += wordAtt_scores
                sent_scores_list += sentAtt_scores
                
                pred_list += predictions.cpu().tolist()
                labels_list += labels.cpu().tolist()
                sent_list += sentences

        # model.train()
        preds = preds.numpy()
        targets = targets.numpy()
        acc = (preds == targets).mean()
        
        return np.mean(losses), acc, word_scores_list, sent_scores_list, sent_list, pred_list, labels_list


### __Train__

In [36]:
train_loss_history = []
val_loss_history = []
best_val_acc = 0

for epoch in range(args.num_epochs):
    model.train()
    for data in tqdm(train_loader):
        # Limpiamos basura de la memoria GPU
        torch.cuda.empty_cache()

        input_docs, labels, sentences = data
        input_docs, labels = input_docs.to(device), labels.to(device)

        # Pasamos los datos por el modelo
        output, _, _ = model(input_docs)
        output = F.log_softmax(output, dim=1)

        loss = criterion(output, labels)
        # Reiniciamos el cálculo del gradiente
        optimizer.zero_grad()
        # Calculamos el gradiente de la pérdida
        loss.backward()
        # Realizamos un paso de la optimización
        optimizer.step()

    # Evaluamos el modelo en los conjuntos de entrenamiento y validación
    train_loss, train_acc, _, _, _, _, _ = eval_model(model, train_loader, criterion, device)
    val_loss, val_acc, _, _, _, _, _ = eval_model(model, val_loader, criterion, device)

    train_loss_history.append(train_loss)
    val_loss_history.append(val_loss)

    print('epoch: %d'%(epoch+1))
    print('train_loss: %5f | val_loss: %5f | train_acc: %5f | val_acc: %5f'%(train_loss, val_loss, train_acc, val_acc))
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state_dict = copy.deepcopy(model.state_dict())

  0%|          | 0/840 [00:00<?, ?it/s]

  0%|          | 0/840 [00:00<?, ?it/s]

  0%|          | 0/210 [00:00<?, ?it/s]

epoch: 1
train_loss: 0.461013 | val_loss: 0.538720 | train_acc: 0.870238 | val_acc: 0.838095


  0%|          | 0/840 [00:00<?, ?it/s]

...

epoch: 2
train_loss: 0.329591 | val_loss: 0.452385 | train_acc: 0.902679 | val_acc: 0.867857


  0%|          | 0/840 [00:00<?, ?it/s]

  0%|          | 0/840 [00:00<?, ?it/s]

  0%|          | 0/210 [00:00<?, ?it/s]

epoch: 3
train_loss: 0.231890 | val_loss: 0.427029 | train_acc: 0.929762 | val_acc: 0.875000


In [138]:
model.load_state_dict(best_state_dict)

train_loss, train_acc, train_w_scores, train_s_scores, train_sents, train_pred, train_labels = eval_model(model, train_loader, criterion, device)
val_loss, val_acc, val_w_scores, val_s_scores, val_sents, val_pred, val_labels = eval_model(model, val_loader, criterion, device)

print('train_loss: %5f | train_acc: %5f'%(train_loss, train_acc))
print('val_loss: %5f | val_acc: %5f'%(val_loss, val_acc))

  0%|          | 0/420 [00:00<?, ?it/s]

  0%|          | 0/105 [00:00<?, ?it/s]

train_loss: 0.040759 | train_acc: 0.986012
val_loss: 0.432752 | val_acc: 0.904762


Podemos observar qu eel modelo se sobreajusta un poco, pero logra ibtener un buen rendimiento a la hora de clasificar, para este tipo de tarea (autor profiling) el modelo con atencion jerarquica logra aprender a clasificar correctamente con un __acc: 0.90__ en el conjunto de validacion. 

## 3. Visualizaciones 

In [292]:
from IPython.display import display, HTML
import matplotlib.pyplot as plt

# Función para visuzalizar la atención a nivel de palabra, tomada de https://gist.github.com/ihsgnef/f13c35cd46624c8f458a4d23589ac768
def colorize(sentence, word_scores):
    # words es una lista de palabras
    # color_array es un arreglo de números entre 0 y 1 de longitud igual al número de palabras
    cmap = matplotlib.colormaps.get_cmap('YlOrBr')
    tokens = sentence if isinstance(sentence, list) else sentence.split()
    colored_tokens = []
    for token, score in zip(tokens, word_scores):
        # Convertir el score a un color (se toma la parte RGB)
        color = matplotlib.colors.rgb2hex(cmap(score)[:3])
        # Envolver cada token en un span con fondo coloreado
        colored_token = f'<span style="background-color: {color}; padding: 2px;">{token}</span>'
        colored_tokens.append(colored_token)
    return ' '.join(colored_tokens)


In [306]:
inv_label_map = {idx: country for country, idx in dataset_train.label_map.items()}
dicc_w_scores_val = {i: val_w_scores[i * 100:(i + 1) * 100] for i in range(len(val_w_scores) // 100)}

k = 7
muestra = random.sample(range(len(val_pred)), k)

cmap = matplotlib.colormaps.get_cmap('Reds')

for idx in muestra:
    ground_truth = inv_label_map[val_labels[idx]]
    prediction = inv_label_map[val_pred[idx]]
    print(f'GT: {ground_truth}  Predicción: {prediction}')
    
    scores_tweets = val_s_scores[idx]  
    sorted_indices = np.flip(np.argsort(scores_tweets))
    tweets = val_sents[idx] 

    for tweet_index in sorted_indices[:6]:
        attention_score = scores_tweets[tweet_index]
        sentence_attention_color = matplotlib.colors.rgb2hex(cmap(attention_score))
        sentence = tweets[tweet_index]
        sentence_str = colorize(sentence, dicc_w_scores_val[idx][tweet_index][:len(sentence)])
        sentence_html = (f'<div style="border-left: 5px solid {sentence_attention_color}; 'f'padding-left: 10px;">{sentence_str}</div>')
        display(HTML(sentence_html))
    print("\n")


GT: chile  Predicción: peru




GT: chile  Predicción: chile




GT: argentina  Predicción: argentina




GT: peru  Predicción: peru




GT: colombia  Predicción: colombia




GT: chile  Predicción: chile




GT: spain  Predicción: spain


El modelo logra tener un buen desempeño, logra poner atencion en palabras clave como en las entidades nombradas (ejemplo: lima, valencia, etc) y es capaz de distinguir caracteristicas distintivas de las variantes del ideoma español. En algunos usuarios se confunde (como en el primer ejemplo de muestra) ya que utiliza palabras que logran confundir al modelo.

## 4. Preguntas 

1. ¿Qué representación de términos uso el profesor para participar? (Consulte: [link]( http://ceur-ws.org/Vol-1866/paper_109.pdf ))

La propuesta del profesor, denominada **User Specific Representation (USR)**, es una variante del modelo de bolsa de términos que busca modelar la caracterización de los usuarios a través de sus interacciones con otros usuarios. Se basa en la hipótesis distribucional de que los perfiles de usuario (*documents*) reflejan preferencias lingüísticas hacia ciertos tópicos, por ejemplo nosotros usamos mucho la palabra _wey_. 
El objetivo principal de USR es capturar la semántica de las palabras mediante su distribución de ocurrencia en los documentos por usuario, utilizando una variante del ponderado por frecuencia de documento (*df*).  


   
2. ¿Qué usó el primer lugar de la competencia? (Basile, et al.) (Consulte: [link](http://ceur-ws.org/Vol-1866/ ))

El primer lugar de la competencia usó un modelo llamado N-GrAM, basado en un clasificador SVM lineal con n-gramas de palabras (unigramas y bigramas) y n-gramas de caracteres (3 a 5-gramas) como características. Este modelo utilizó ponderación tf-idf con escalado sublineal de la frecuencia de términos, donde en lugar de usar el término de frecuencia $(tf)$ se utilizó: $1 + \log(tf)$. 

Además, se excluyeron los términos que solo eran usados por un autor _(min_df = 2)_ lo cual resulto en una respuesta positiva para el modelo.


3. ¿Cuántos y que competidores usaron deep learning? En una o dos oraciones escriba qué hicieron: [link](https://ceur-ws.org/Vol-1866/)

6 competidores usaron deep learning:

* __Bakhteev-Khazov__: Utilizaron un modelo seq2seq (Encoder-Decoder) utilizando LSTM, utilizaron metodos para cambiar la contenido de las oraciones tratando de salvar la oracion y tambien cambiaron la estructura y longitud de la oraciones.

*   __Marc Franco-Salvador et al__: Utilizaron un modelo basado en Deep Averaging Networks (DAN) con embeddings de subpalabras (n-gramas de caracteres). 
   
*   __Don Kodiyan et al__: Este equipo presentó un modelo basado en una RNN bidireccional utilizando GRUs, combinado con un mecanismo de atención. 

*   __Yasuhide  Miura et al__: Utilizaron modelos de redes neuronales recurrentes y convulucionales que incorporan un mecanismo de atencion, ellos combinan información de palabras y caracteres (para las palabras las RNN y para los carcteres las CNN). 

*   __Nils Schaetti__: Utilizó un enfoque que combinaba un modelo basado en TF-IDF y un modelo de redes neuronales convolucionales (CNN). 

*   __Sebastian  Sierra et al__: Utilizaron modelos por separado para identificar el género y la variedad lingüística de los usuarios utilizando CNN. 


## 5. Atención en jerarquía vs no tenerla

In [323]:
args.use_word_attn = False     
args.use_sentence_attn = False   

model_noatt = HAN(args, embedding_matrix=dataset_train.emb_mat).to(device)
optimizer = optim.Adam(model.parameters(), 
                       lr=args.lr,
                       weight_decay=args.weight_decay, 
                       betas = (args.beta1, args.beta2))
criterion = nn.NLLLoss()

In [324]:
train_loss_history = []
val_loss_history = []
best_val_acc = 0

for epoch in range(args.num_epochs):
    model_noatt.train()
    for data in tqdm(train_loader):
        # Limpiamos basura de la memoria GPU
        torch.cuda.empty_cache()

        input_docs, labels, sentences = data
        input_docs, labels = input_docs.to(device), labels.to(device)

        # Pasamos los datos por el modelo
        output, _, _ = model_noatt(input_docs)
        output = F.log_softmax(output, dim=1)

        loss = criterion(output, labels)
        # Reiniciamos el cálculo del gradiente
        optimizer.zero_grad()
        # Calculamos el gradiente de la pérdida
        loss.backward()
        # Realizamos un paso de la optimización
        optimizer.step()

    # Evaluamos el modelo en los conjuntos de entrenamiento y validación
    train_loss, train_acc, _, _, _, _, _ = eval_model(model_noatt, train_loader, criterion, device)
    val_loss, val_acc, _, _, _, _, _ = eval_model(model_noatt, val_loader, criterion, device)

    train_loss_history.append(train_loss)
    val_loss_history.append(val_loss)

    print('epoch: %d'%(epoch+1))
    print('train_loss: %5f | val_loss: %5f | train_acc: %5f | val_acc: %5f'%(train_loss, val_loss, train_acc, val_acc))
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state_dict = copy.deepcopy(model_noatt.state_dict())

  0%|          | 0/420 [00:00<?, ?it/s]

  0%|          | 0/420 [00:00<?, ?it/s]

  0%|          | 0/105 [00:00<?, ?it/s]

epoch: 1
train_loss: 2.029308 | val_loss: 2.012060 | train_acc: 0.159821 | val_acc: 0.167857


  0%|          | 0/420 [00:00<?, ?it/s]

...

epoch: 2
train_loss: 2.019786 | val_loss: 2.002226 | train_acc: 0.158333 | val_acc: 0.157143


  0%|          | 0/420 [00:00<?, ?it/s]

  0%|          | 0/420 [00:00<?, ?it/s]

  0%|          | 0/105 [00:00<?, ?it/s]

epoch: 3
train_loss: 2.021676 | val_loss: 2.005893 | train_acc: 0.158929 | val_acc: 0.173810


In [325]:
model_noatt.load_state_dict(best_state_dict)

train_loss, train_acc, train_w_scores, train_s_scores, train_sents, train_pred, train_labels = eval_model(model_noatt, train_loader, criterion, device)
val_loss, val_acc, val_w_scores, val_s_scores, val_sents, val_pred, val_labels = eval_model(model_noatt, val_loader, criterion, device)

print('train_loss: %5f | train_acc: %5f'%(train_loss, train_acc))
print('val_loss: %5f | val_acc: %5f'%(val_loss, val_acc))

  0%|          | 0/420 [00:00<?, ?it/s]

  0%|          | 0/105 [00:00<?, ?it/s]

train_loss: 2.021768 | train_acc: 0.163393
val_loss: 2.005893 | val_acc: 0.173810


El modelo sin atención tiene un rendimiento bajo, no logra ajustarce a los datos ya que los textos son informales. Además se utiliza mean pooling que toma el promedio de los inputs, lo que no ayuda a encontrar patrones en los datos.  

In [37]:
class CBTN(nn.Module):
    def __init__(self, emb_mat, num_classes=7, emb_dim=100, ff_dim=512,
                 word_layers=2, sentence_layers=1, dropout=0.3):
        super(CBTN, self).__init__()

        self.embedding = nn.Embedding.from_pretrained(
            embeddings=torch.FloatTensor(emb_mat),
            freeze=True
        )

        self.word_transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=emb_dim, nhead=4,
                                       dim_feedforward=ff_dim, dropout=dropout,
                                       batch_first=True),
            num_layers=word_layers
        )

        self.sentence_transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=emb_dim, nhead=4,
                                       dim_feedforward=ff_dim, dropout=dropout,
                                       batch_first=True),
            num_layers=sentence_layers
        )

        self.classifier = nn.Sequential(
            nn.Linear(emb_dim, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):  # x: (B, S, T)
        B, S, T = x.size()

        # Paso 1: Lookup de embeddings
        x = self.embedding(x)  # (B, S, T, D)

        # Paso 2: Aplica atención por palabra (por oración)
        x = x.view(-1, T, x.size(-1))  # (B*S, T, D)
        word_encoded = self.word_transformer(x)  # (B*S, T, D)
        word_repr = word_encoded.mean(dim=1)     # (B*S, D)

        # Paso 3: Agrupar por usuario
        sent_input = word_repr.view(B, S, -1)     # (B, S, D)

        # Paso 4: Atención a nivel de oración
        sent_encoded = self.sentence_transformer(sent_input)  # (B, S, D)
        doc_repr = sent_encoded.mean(dim=1)                   # (B, D)

        # Paso 5: Clasificación
        return self.classifier(doc_repr)                      # (B, num_classes)



In [38]:
def eval_model(model, dataloader, criterion, device):
    '''Función para evaluar el modelo.'''
    model.eval()
    losses = []
    preds = torch.empty(0).long()
    targets = torch.empty(0).long()
    pred_list = []
    labels_list = []
    
    with torch.no_grad():
        for batch_tensor, labels, _ in tqdm(dataloader):
            torch.cuda.empty_cache()
            batch_tensor, labels = batch_tensor.to(device), labels.to(device)

            # Output del modelo (sin atención)
            logits = model(batch_tensor)  # (batch_size, num_classes)
            output = F.log_softmax(logits, dim=1)

            loss = criterion(output, labels)
            losses.append(loss.item())

            predictions = output.argmax(1)
            preds = torch.cat([preds, predictions.cpu()], dim=0)
            targets = torch.cat([targets, labels.cpu()], dim=0)
            pred_list += predictions.cpu().tolist()
            labels_list += labels.cpu().tolist()

    acc = (preds.numpy() == targets.numpy()).mean()
    return np.mean(losses), acc, pred_list, labels_list

In [41]:
model = CBTN(emb_mat=dataset_train.emb_mat).to(device)
optimizer = optim.Adam(model.parameters(), 
                       lr=args.lr,
                       weight_decay=args.weight_decay, 
                       betas = (args.beta1, args.beta2))
criterion = nn.NLLLoss()

In [43]:
train_loss_history = []
val_loss_history = []
best_val_acc = 0

# 1. Asegúrate de que emb_mat sea tensor en CUDA desde el principio
emb_mat = torch.tensor(dataset_train.emb_mat, dtype=torch.float32).to(device)

for epoch in range(args.num_epochs):
    model.train()

    for batch_tensor, labels, _ in tqdm(train_loader):
        torch.cuda.empty_cache()
        batch_tensor = batch_tensor.to(device)  # (B, S, T)
        labels = labels.to(device)
    
        logits = model(batch_tensor)  # modelo hace embedding internamente
        output = F.log_softmax(logits, dim=1)

        
        loss = criterion(output, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    # Evaluar después de cada epoch
    train_loss, train_acc, _, _ = eval_model(model, train_loader, criterion, device)
    val_loss, val_acc, _, _ = eval_model(model, val_loader, criterion, device)

    train_loss_history.append(train_loss)
    val_loss_history.append(val_loss)

    print(f'Epoch {epoch + 1}')
    print(f'Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state_dict = copy.deepcopy(model.state_dict())

  0%|          | 0/840 [00:00<?, ?it/s]

C:\Users\ericl\anaconda3\envs\nlp\Lib\site-packages\torch\nn\functional.py:5560: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = scaled_dot_product_attention(q, k, v, attn_mask, dropout_p, is_causal)


  0%|          | 0/840 [00:00<?, ?it/s]

  0%|          | 0/210 [00:00<?, ?it/s]

Epoch 1
Train Loss: 1.9221 | Val Loss: 1.9171 | Train Acc: 0.2018 | Val Acc: 0.2036


...

Epoch 2
Train Loss: 1.9461 | Val Loss: 1.9461 | Train Acc: 0.1429 | Val Acc: 0.1429


  0%|          | 0/840 [00:00<?, ?it/s]

  0%|          | 0/840 [00:00<?, ?it/s]

  0%|          | 0/210 [00:00<?, ?it/s]

Epoch 3
Train Loss: 1.8798 | Val Loss: 1.8876 | Train Acc: 0.2205 | Val Acc: 0.2143


In [56]:
# ----------- collate_fn -------------
import torch
from torch.nn.utils.rnn import pad_sequence

def hier_collate(batch: List[Dict[str, Any]], pad_id: int):
    """
    Devuelve:
        flat_input_ids      : (B·T, L)
        flat_attention_mask : (B·T, L)
        tweet_mask          : (B, T)  1 = tweet válido, 0 = padding
        labels              : (B,)
        shape_info          : dict -> nº tweets por user (para re-agrupar)
    """
    labels = torch.tensor([item["label"] for item in batch], dtype=torch.long)

    # 1) contar tweets por usuario y obtener máximos del minibatch
    num_tweets  = [sum(len(block) for block in itm["blocks"]) for itm in batch]
    max_tweets  = max(num_tweets)

    flat_input_ids      = []
    flat_attention_mask = []
    tweet_masks         = []

    for itm in batch:
        tweets_tokenized = [t for block in itm["blocks"] for t in block]        # aplanar
        n = len(tweets_tokenized)

        # padding de tweets hasta max_tweets
        for _ in range(max_tweets - n):
            tweets_tokenized.append({"input_ids": [pad_id], "attention_mask": [0]})

        # ahora tokenizar cada tweet; pad_sequence a nivel de sub-tokens
        ids  = [torch.tensor(t["input_ids"])      for t in tweets_tokenized]
        am   = [torch.tensor(t["attention_mask"]) for t in tweets_tokenized]

        ids  = pad_sequence(ids, batch_first=True, padding_value=pad_id)
        am   = pad_sequence(am,  batch_first=True, padding_value=0)

        flat_input_ids.append(ids)
        flat_attention_mask.append(am)
        tweet_masks.append([1]*n + [0]*(max_tweets-n))

    # concatenar: tamaño -> (B, T, L)
    input_ids  = torch.stack(flat_input_ids)       # (B, T, L)
    attn_mask  = torch.stack(flat_attention_mask)  # (B, T, L)
    tweet_mask = torch.tensor(tweet_masks, dtype=torch.bool)  # (B, T)

    # aplanar tweets para enviar a BETO
    B, T, L = input_ids.shape
    flat_input_ids      = input_ids.reshape(B*T, L)
    flat_attention_mask = attn_mask.reshape(B*T, L)

    return {
        "input_ids": flat_input_ids,
        "attention_mask": flat_attention_mask,
        "tweet_mask": tweet_mask,
        "labels": labels,
        "tweets_per_user": torch.tensor(num_tweets)
    }



In [57]:
dff
# train_Dataset = AutorProfilingHierDataset()

NameError: name 'dff' is not defined

In [45]:
train_loader = DataLoader(dataset_train, 
                          batch_size=args.batch_size, 
                          collate_fn = collate_fn, 
                          shuffle=True)

val_loader = DataLoader(dataset_val, 
                        batch_size=args.batch_size, 
                        collate_fn = collate_fn, 
                        shuffle=False)